# Dataset builder
This contains multiple blocks of code, not meant to be run together. It is a record of how I made my data

### Attempt 1: Fairprice stock images

In [ ]:
# After manually downloading images from fairprice and removing the backgrounds using https://removal.ai/
from pathlib import Path
from PIL import Image

folder_path = Path("Training Data")
items = {}
for folder in folder_path.iterdir():
    temp = []
    for i in Path(folder).iterdir():
        temp.append(Path(i))
    items[folder] = temp
x = list(items.keys())
for i in x:
    for j in items[i]:
        output_path = str(i / f"{j.stem}_cropped.png")
        with Image.open(j) as im:
            bbox = im.getbbox()
            if bbox:
                cropped_img = im.crop(bbox)
                cropped_img.save(output_path)

#### Fairprice images worked under good conditions, but shadows and new backgrounds were too hard for it. Instead, I decided to take my own photos from multiple angles in combination with the existing fairprice images to train the model using.

#### In order to remove images, instead of doing manual work, i decided to try using open source options. I used rembg with birefnet-general to remove all the backgrounds. I then manually inspected and then moved them to the correct folder

In [ ]:
from rembg import remove, new_session
from PIL import Image

session = new_session("birefnet-general") # selects model used for background removal

#### Loop through my folder and remove backgrounds

In [ ]:
from pathlib import Path

for file in Path(r"New training data").iterdir():
    input_path = str(file)
    output_path = str(file.parent / (file.stem + ".out.png"))

    with open(input_path, 'rb') as i:
        with open(output_path, 'wb') as o:
            inp = i.read()
            output = remove(inp, session=session)
            o.write(output)

### Once everything looked good, I needed to remove the excess transparent pixels using crop + bbox

In [ ]:
files_list = list(Path(r"New training data").glob("*.out.png")) 
for file in files_list:
    out_dir = Path(r"New training data")
    output_path = str(out_dir / f"{file.stem}_cropped.png")
    with Image.open(file) as im:
        # get the alpha channel (0-255 transparency) and turns everything below 10 to zero, rembg does not give clean (0,0,0,0) pixels after background removal
        bbox = im.getchannel("A").point(lambda a: 255 if a > 10 else 0).getbbox() 
        if bbox:
            cropped_img = im.crop(bbox)
            cropped_img.save(output_path)

#### After manually moving the images to their correct folders, I needed to rename them nicely

In [ ]:
for folder in Path(r"Training Data").iterdir():
    # renames everything to a temporary name to avoid conflicts
    count = 0
    for image in list(folder.iterdir()):
        image.rename(folder / f"temp_{count}.png")
        count += 1

    # renames to proper name
    count = 0
    for image in list(folder.iterdir()):
        image.rename(folder / f"image_{count}.png")
        count += 1